# 📈 Stock Price Prediction with LSTM + Attention

**Project Overview:**
- Built an LSTM neural network with Attention mechanism for stock price forecasting
- Multi-step prediction (predict next 5 days)
- Technical indicators as features (SMA, EMA, RSI, MACD)
- Explainable AI through attention weights visualization

**Key Skills Demonstrated:**
- Time Series Analysis & Forecasting
- Recurrent Neural Networks (LSTM)
- Attention Mechanisms
- Financial Data Processing
- Feature Engineering

In [ ]:
# 📦 Imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 📊 Data Generation (Simulated Stock Data)

In [ ]:
def generate_stock_data(days=1000, start_price=100, volatility=0.02, trend=0.0005):
    """
    Generate realistic synthetic stock data with trend and volatility
    In production, use: yfinance, pandas_datareader, or Alpha Vantage API
    """
    np.random.seed(42)
    
    dates = pd.date_range(end=datetime.now(), periods=days)
    
    # Generate price using geometric Brownian motion
    returns = np.random.normal(trend, volatility, days)
    prices = start_price * np.exp(np.cumsum(returns))
    
    # Generate OHLC data
    df = pd.DataFrame({
        'Date': dates,
        'Open': prices * (1 + np.random.normal(0, 0.005, days)),
        'High': prices * (1 + np.abs(np.random.normal(0, 0.01, days))),
        'Low': prices * (1 - np.abs(np.random.normal(0, 0.01, days))),
        'Close': prices,
        'Volume': np.random.randint(1000000, 10000000, days)
    })
    
    df.set_index('Date', inplace=True)
    return df

# Generate dataset
df = generate_stock_data(days=1500)
print(f"Dataset shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nDate range: {df.index[0]} to {df.index[-1]}")

## 🔧 Technical Indicators (Feature Engineering)

In [ ]:
def add_technical_indicators(df):
    """Add technical analysis indicators as features"""
    df = df.copy()
    
    # Simple Moving Averages
    df['SMA_10'] = df['Close'].rolling(window=10).mean()
    df['SMA_30'] = df['Close'].rolling(window=30).mean()
    
    # Exponential Moving Average
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
    
    # MACD
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    
    # RSI (Relative Strength Index)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    df['BB_Middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Middle'] + (bb_std * 2)
    df['BB_Lower'] = df['BB_Middle'] - (bb_std * 2)
    
    # Price change and volatility
    df['Price_Change'] = df['Close'].pct_change()
    df['Volatility'] = df['Price_Change'].rolling(window=10).std()
    
    # Volume features
    df['Volume_SMA'] = df['Volume'].rolling(window=10).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']
    
    return df.dropna()

# Add indicators
df = add_technical_indicators(df)
print(f"Features after engineering: {df.shape[1]}")
print(f"\nFeatures: {list(df.columns)}")
df.head()

## 🔗 LSTM + Attention Model

In [ ]:
class Attention(nn.Module):
    """
    Attention Mechanism for LSTM
    Helps model focus on important time steps
    """
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.hidden_dim = hidden_dim
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, lstm_output):
        # lstm_output: (batch, seq_len, hidden_dim)
        attention_weights = self.attention(lstm_output).squeeze(-1)  # (batch, seq_len)
        attention_weights = torch.softmax(attention_weights, dim=1)  # Normalize
        
        # Apply attention weights
        context = torch.bmm(attention_weights.unsqueeze(1), lstm_output).squeeze(1)
        return context, attention_weights

class LSTMStockPredictor(nn.Module):
    """
    LSTM with Attention for Stock Price Prediction
    """
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, output_dim=5, dropout=0.2):
        super(LSTMStockPredictor, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Attention mechanism
        self.attention = Attention(hidden_dim)
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        
        # LSTM forward
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, hidden_dim)
        
        # Apply attention
        context, attention_weights = self.attention(lstm_out)
        
        # Final prediction
        output = self.fc(context)  # (batch, output_dim)
        
        return output, attention_weights

# Test model
input_dim = 15  # Number of features
model = LSTMStockPredictor(input_dim=input_dim, hidden_dim=128, num_layers=2, output_dim=5).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print("✅ LSTM + Attention model loaded successfully")

## 📚 Data Preparation

In [ ]:
class StockDataset(Dataset):
    """Stock price dataset for time series"""
    def __init__(self, data, seq_length=60, pred_length=5):
        self.data = data
        self.seq_length = seq_length
        self.pred_length = pred_length
    
    def __len__(self):
        return len(self.data) - self.seq_length - self.pred_length
    
    def __getitem__(self, idx):
        # Input sequence
        x = self.data[idx:idx + self.seq_length]
        
        # Target: next 5 days close prices
        y = self.data[idx + self.seq_length:idx + self.seq_length + self.pred_length, 3]  # Close price index
        
        return torch.FloatTensor(x), torch.FloatTensor(y)

# Prepare data
feature_columns = ['Open', 'High', 'Low', 'Close', 'Volume',
                  'SMA_10', 'SMA_30', 'EMA_12', 'EMA_26',
                  'MACD', 'MACD_Signal', 'RSI',
                  'BB_Upper', 'BB_Lower', 'Volume_Ratio']

# Scale features
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[feature_columns])

# Separate scaler for close price (for inverse transform)
close_scaler = MinMaxScaler()
close_scaler.fit(df[['Close']])

# Create dataset
seq_length = 60  # Look back 60 days
pred_length = 5  # Predict next 5 days

dataset = StockDataset(scaled_data, seq_length, pred_length)

# Train/Val/Test split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

# Data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Sequence length: {seq_length} days")
print(f"Prediction horizon: {pred_length} days")

## 🎯 Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=100, patience=15):
    """Training loop with early stopping"""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            outputs, _ = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                
                outputs, _ = model(x_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_lstm_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    return history

print("\n🚀 Starting training...")
history = train_model(model, train_loader, val_loader, num_epochs=100, patience=15)

## 📊 Evaluation & Visualization

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_lstm_model.pth'))

# Test predictions
model.eval()
all_preds = []
all_actuals = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        outputs, _ = model(x_batch)
        
        all_preds.append(outputs.cpu().numpy())
        all_actuals.append(y_batch.numpy())

all_preds = np.vstack(all_preds)
all_actuals = np.vstack(all_actuals)

# Inverse transform to get actual prices
preds_inv = close_scaler.inverse_transform(all_preds)
actuals_inv = close_scaler.inverse_transform(all_actuals)

# Calculate metrics
mse = mean_squared_error(actuals_inv.flatten(), preds_inv.flatten())
rmse = np.sqrt(mse)
mae = mean_absolute_error(actuals_inv.flatten(), preds_inv.flatten())
mape = np.mean(np.abs((actuals_inv - preds_inv) / actuals_inv)) * 100
r2 = r2_score(actuals_inv.flatten(), preds_inv.flatten())

print("\n📈 Test Set Performance:")
print(f"   RMSE: ${rmse:.2f}")
print(f"   MAE: ${mae:.2f}")
print(f"   MAPE: {mape:.2f}%")
print(f"   R² Score: {r2:.4f}")

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Training history
axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Training History')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Predictions vs Actual (Day 1)
axes[0, 1].scatter(actuals_inv[:, 0], preds_inv[:, 0], alpha=0.5)
axes[0, 1].plot([actuals_inv[:, 0].min(), actuals_inv[:, 0].max()], 
                [actuals_inv[:, 0].min(), actuals_inv[:, 0].max()], 'r--')
axes[0, 1].set_title('Predictions vs Actual (Day 1)')
axes[0, 1].set_xlabel('Actual Price ($)')
axes[0, 1].set_ylabel('Predicted Price ($)')

# Time series plot
sample_idx = slice(0, 50)
axes[1, 0].plot(actuals_inv[sample_idx, 0], label='Actual', linewidth=2)
axes[1, 0].plot(preds_inv[sample_idx, 0], label='Predicted', linewidth=2, alpha=0.8)
axes[1, 0].set_title('Price Prediction - Time Series')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_ylabel('Price ($)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Multi-day prediction visualization
sample = 0
days = range(1, 6)
axes[1, 1].plot(days, actuals_inv[sample], 'o-', label='Actual', linewidth=2, markersize=8)
axes[1, 1].plot(days, preds_inv[sample], 's-', label='Predicted', linewidth=2, markersize=8)
axes[1, 1].set_title('Multi-Day Prediction (Sample)')
axes[1, 1].set_xlabel('Days Ahead')
axes[1, 1].set_ylabel('Price ($)')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('stock_prediction_results.png', dpi=150)
plt.show()

## 🔍 Attention Visualization (Explainability)

In [ ]:
# Visualize attention weights
model.eval()

# Get a sample
x_sample, y_sample = test_dataset[0]
x_sample = x_sample.unsqueeze(0).to(device)

with torch.no_grad():
    pred, attention_weights = model(x_sample)

attention_weights = attention_weights.cpu().numpy()[0]

# Plot attention over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Input sequence (Close price)
close_prices = x_sample[0, :, 3].cpu().numpy()  # Close price is index 3
close_prices = close_scaler.inverse_transform(close_prices.reshape(-1, 1)).flatten()

axes[0].plot(close_prices, linewidth=2, color='blue')
axes[0].set_title('Input Sequence (60 Days Close Price)')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True)

# Attention weights
axes[1].bar(range(len(attention_weights)), attention_weights, color='orange')
axes[1].set_title('Attention Weights (What the model focuses on)')
axes[1].set_xlabel('Time Step (Days)')
axes[1].set_ylabel('Attention Weight')
axes[1].grid(True)

plt.tight_layout()
plt.savefig('attention_visualization.png', dpi=150)
plt.show()

print(f"\n🔍 Model Prediction: ${close_scaler.inverse_transform(pred[0, 0].cpu().numpy().reshape(1, -1))[0, 0]:.2f}")
print(f"   Actual: ${close_scaler.inverse_transform(y_sample[0].numpy().reshape(1, -1))[0, 0]:.2f}")
print(f"\n💡 Top 5 most attended time steps: {np.argsort(attention_weights)[-5:][::-1]}")

## ✅ Summary

In [ ]:
print("\n" + "="*60)
print("✅ PROJECT 2 COMPLETE: LSTM Stock Prediction with Attention")
print("="*60)
print("\n📝 Key Achievements:")
print("   • Built LSTM with Attention mechanism from scratch")
print("   • Multi-step forecasting (5 days ahead)")
print("   • Technical indicators feature engineering")
print("   • Model explainability via attention visualization")
print("   • Production-ready evaluation metrics")
print(f"\n📊 Performance Metrics:")
print(f"   • RMSE: ${rmse:.2f}")
print(f"   • MAPE: {mape:.2f}%")
print(f"   • R² Score: {r2:.4f}")
print("\n🔧 Technical Stack:")
print("   • PyTorch for deep learning")
print("   • LSTM + Attention architecture")
print("   • Feature engineering (RSI, MACD, Bollinger Bands)")
print("   • Early stopping & learning rate scheduling")
print("="*60)